# SRGAN ×3 — 학습된 GAN 적용과 판별자 붕괴

흐린 위성사진(10 m) → 3배 선명하게(3.33 m). 미리 학습해둔 SRGAN Generator 를 쓴다.
학습 로그로 **판별자가 무너지는 과정**을 함께 본다. GPU 없어도 됩니다.

## 1. 데이터

In [ ]:
import urllib.request
LIB = 'https://raw.githubusercontent.com/BWMIN-Hub/SR_practice/main/lib'
for m in ['sr_utils.py', 'srgan_models.py', 'srgan_losses.py']:
    urllib.request.urlretrieve(f'{LIB}/{m}', m)

from sr_utils import *

val_lr, val_hr = pair('validation', REP['validation'])
test_lr = load_test()
show([('validation (Paris)', val_lr, val_hr), ('test (Incheon)', test_lr, None)])

## 2. 학습

**코드가 도는지 확인하는 용도다.** 적은 데이터로 몇 번만 돌린다.
실제 결과는 다음 단계에서 전체 데이터로 학습해둔 가중치로 본다.

생성자와 판별자를 번갈아 갱신하는 것이 GAN 학습의 전부다.

In [ ]:
import torch
from torch.utils.data import DataLoader, TensorDataset
from srgan_models import Generator, Discriminator
from srgan_losses import GeneratorLoss

N_TRAIN, EPOCHS, BATCH = 16, 3, 4
dev = 'cuda' if torch.cuda.is_available() else 'cpu'

lo, hi = zip(*[pair('training', s) for s in list_split('training')[:N_TRAIN]])
to_t = lambda a: torch.from_numpy(np.stack(a).transpose(0, 3, 1, 2)).float() / 255
loader = DataLoader(TensorDataset(to_t(lo), to_t(hi)), batch_size=BATCH, shuffle=True)

netG, netD = Generator(3).to(dev).train(), Discriminator().to(dev).train()
optG, optD = torch.optim.Adam(netG.parameters()), torch.optim.Adam(netD.parameters())
crit = GeneratorLoss().to(dev)

for ep in range(1, EPOCHS + 1):
    dl = gl = dx = dgz = 0.0
    for x, y in loader:
        x, y = x.to(dev), y.to(dev)
        fake = netG(x)                                   # 1) 생성자
        g_loss = crit(netD(fake).mean(), fake, y)
        optG.zero_grad(); g_loss.backward(); optG.step()

        real_out, fake_out = netD(y).mean(), netD(fake.detach()).mean()
        d_loss = 1 - real_out + fake_out                 # 2) 판별자
        optD.zero_grad(); d_loss.backward(); optD.step()

        gl += g_loss.item(); dl += d_loss.item()
        dx += real_out.item(); dgz += fake_out.item()
    n = len(loader)
    print(f'epoch {ep}/{EPOCHS}  Loss_G {gl/n:.4f}  Loss_D {dl/n:.4f}  '
          f'D(x) {dx/n:.3f}  D(G(z)) {dgz/n:.3f}')

print(f'\n{N_TRAIN}장 x {EPOCHS}회 — 동작 확인용이다. 실제 성능은 아래 가중치로 본다.')

## 3. 미리 학습된 가중치 적용

In [ ]:
import torch
from srgan_models import load_srgan

MODEL = f'{BASE}/models/02_srgan_x3'
net = load_srgan(fetch(f'{MODEL}/checkpoints/srgan_g_x3_ep70.pth', 'srgan_x3.pth'))

@torch.no_grad()
def upscale(lr):
    t = torch.from_numpy(lr.transpose(2, 0, 1)).float()[None].to(next(net.parameters()).device) / 255
    out = net(t).clamp(0, 1)[0].cpu().numpy().transpose(1, 2, 0) * 255
    return out.round().astype('uint8')

print('SRGAN x3 (epoch 70) 준비 완료')

## 4. 정량 평가

In [ ]:
rows = compare(upscale, label='SRGAN')

## 5. 결과

In [ ]:
zoom([('Bicubic', bicubic(val_lr)), ('SRGAN', upscale(val_lr)), ('Target HR', val_hr)],
     title='validation (Paris)')

test_sr = upscale(test_lr)
zoom([('Bicubic', bicubic(test_lr)), ('SRGAN', test_sr)], title='test (Incheon) - no target')
imageio.imwrite('incheon_srgan.png', test_sr)

## 6. 판별자는 어떻게 무너졌나

학습 중 기록한 로그를 본다. 배율만 다른 두 학습(×2 / ×3)에서
**정반대 방향으로** 붕괴가 일어났다.

In [ ]:
import pandas as pd

E = {}
for k in [2, 3]:
    p = fetch(f'{MODEL}/statistics/x{k}_train_results.csv', f'x{k}.csv')
    E[k] = pd.read_csv(p, index_col=0)

C = {2: '#c96a5b', 3: '#2f6f9f'}
fig, ax = plt.subplots(1, 3, figsize=(16, 4.2))

for j, k in enumerate([2, 3]):
    e = E[k]
    ax[j].plot(e.index, e.Score_D, color='#2f6f9f', lw=1.7, label='D(x)  real')
    ax[j].plot(e.index, e.Score_G, color='#c96a5b', lw=1.7, label='D(G(z))  fake')
    ax[j].axhline(.5, ls='--', c='#888', lw=1)
    ax[j].set_title(f'x{k}: collapses to {0 if k == 2 else 1}'
                    f'  ("everything is {"FAKE" if k == 2 else "REAL"}")')
    ax[j].set_xlabel('epoch'); ax[j].set_ylim(-.05, 1.08)
    ax[j].grid(alpha=.3); ax[j].legend(fontsize=8)

for k in [2, 3]:
    ax[2].plot(E[k].index, E[k].Loss_D, color=C[k], lw=1.7, label=f'x{k}')
ax[2].axhline(1.0, ls='--', c='#888', lw=1)
ax[2].set_title('Discriminator loss — both flatten at 1.0')
ax[2].set_xlabel('epoch'); ax[2].set_ylim(.85, 1.06)
ax[2].grid(alpha=.3); ax[2].legend(fontsize=8)
plt.tight_layout(); plt.show()

for k in [2, 3]:
    e = E[k]
    print(f'x{k}: 마지막 D(x)={e.iloc[-1].Score_D:.4f}  D(G(z))={e.iloc[-1].Score_G:.4f}  '
          f'Loss_D={e.iloc[-1].Loss_D:.4f}')

`Loss_D = 1 - D(x) + D(G(z))` 는 둘 다 0일 때도, 둘 다 1일 때도 정확히 1.0 이다.
**손실 그래프만으로는 두 붕괴를 구분할 수 없다.**

In [ ]:
for k in [2, 3]:
    e = E[k]
    fig, ax = plt.subplots(figsize=(7, 2.6))
    ax.plot(e.index, e.w_adversarial, color=C[k], lw=1.8)
    ax.set_title(f'x{k}: weighted adversarial term (0.001 x adversarial)')
    ax.set_xlabel('epoch'); ax.grid(alpha=.3)
    plt.tight_layout(); plt.show()
    v = e.w_adversarial
    print(f'x{k}: {v.iloc[0]:.6f} -> {v.iloc[-1]:.6f}   '
          f'{"1.0 로 포화 (D 가 전부 가짜로 판정)" if v.iloc[-1] > 5e-4 else "0 으로 포화 (D 가 전부 진짜로 판정)"}')
    print('   상수가 되면 기울기가 0 이라 생성자는 적대적 신호를 못 받는다.\n')